# 03 – Dummy-Classifier-Baseline

Dieses Notebook bestimmt eine einfache Referenzleistung für die nachfolgenden
Klassifikationsmodelle. Der DummyClassifier ignoriert die Featurewerte und
erstellt Vorhersagen ausschließlich anhand der Klassenverteilung.

Die Bewertung erfolgt mit einer geschichteten 5-Fold-Cross-Validation auf den
Trainingsdaten. Verwendet werden PR-AUC, ROC-AUC, Precision, Recall und F1.
Die Ergebnisse werden gespeichert, damit alle späteren Modelle mit derselben
Baseline verglichen werden können. Die Testdaten werden nicht verwendet.

## 1. Bibliotheken und Projektpfade

Zunächst werden die benötigten Bibliotheken importiert und die zentralen
Projektpfade definiert.

In [1]:
from pathlib import Path 
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate 
from sklearn.metrics import (
    make_scorer,
    precision_score,
    recall_score,
    f1_score
)

In [2]:
project_root = Path.cwd()
processed_dir = project_root / "data" / "processed"

## 2. Laden der Trainingsdaten

Für die Cross-Validation werden ausschließlich die zentral vorbereiteten,
unskalierten Trainingsdaten geladen. Der gespeicherte Testdatensatz bleibt
bis zur finalen Evaluation des ausgewählten Modells unangetastet.

In [3]:
train_data = pd.read_parquet(
    processed_dir / "train.parquet"
)

test_data = pd.read_parquet(
    processed_dir / "test.parquet"
)

print("Trainingsdaten:", train_data.shape)
print("Testdaten:", test_data.shape)

Trainingsdaten: (160000, 201)
Testdaten: (40000, 201)


## 3. Trennung von Features und Zielvariable

Die 200 anonymisierten Merkmale werden als Featurematrix `X_train` verwendet.
Die Spalte `target` bildet die binäre Zielvariable `y_train`.

In [4]:
X_train = train_data.drop(columns=["target"])
y_train = train_data["target"]

X_test = test_data.drop(columns=["target"])
y_test = test_data["target"]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (160000, 200)
y_train: (160000,)
X_test: (40000, 200)
y_test: (40000,)


## 4. Definition der Evaluationsmetriken

In [5]:
scoring = {
    "PR_AUC" : "average_precision" ,
    "ROC_AUC": "roc_auc" ,
    "Precision" : make_scorer(
        precision_score,
        zero_division=0 
    ),
    "Recall" : make_scorer(
        recall_score,
        zero_division=0 
    ),
    "F1": make_scorer(
        f1_score,
        zero_division=0
    )
}

## 5. Cross-Validation der Dummy-Baseline

In [6]:
dummy_model = DummyClassifier(
    strategy = "prior"
)
dummy_results = cross_validate(
    dummy_model,
    X_train,
    y_train,
    cv = StratifiedKFold(
        n_splits= 5 , 
        shuffle = True,
        random_state = 42
    ),
    scoring = scoring 
)

for metric in scoring:
    mean_score = dummy_results[f"test_{metric}"].mean()
    print(f"{metric}: {mean_score:.4f}")

PR_AUC: 0.1005
ROC_AUC: 0.5000
Precision: 0.0000
Recall: 0.0000
F1: 0.0000


## Speicherung der Baseline-Ergebnisse

Die Ergebnisse der fünf Cross-Validation-Folds werden gespeichert. Dadurch
können später Mittelwert und Streuung der Dummy-Baseline mit den Ergebnissen
der trainierten Modelle verglichen werden.

In [7]:
results_dir = project_root / "results" / "cv"

results_dir.mkdir(
    parents=True,
    exist_ok=True
)

In [8]:
n_folds = len(dummy_results["test_PR_AUC"])

dummy_scores = pd.DataFrame({
    "Modell": ["DummyClassifier"] * n_folds,
    "Fold": range(1, n_folds + 1),
    "PR_AUC": dummy_results["test_PR_AUC"],
    "ROC_AUC": dummy_results["test_ROC_AUC"],
    "Precision": dummy_results["test_Precision"],
    "Recall": dummy_results["test_Recall"],
    "F1": dummy_results["test_F1"],
    "Fit_Time": dummy_results["fit_time"],
    "Score_Time": dummy_results["score_time"]
})

dummy_scores

,Modell,Fold,PR_AUC,ROC_AUC,Precision,Recall,F1,Fit_Time,Score_Time
0,DummyClassifier,1,0.100469,0.5,0.0,0.0,0.0,0.113831,0.018236
1,DummyClassifier,2,0.100469,0.5,0.0,0.0,0.0,0.082810,0.014472
2,DummyClassifier,3,0.100500,0.5,0.0,0.0,0.0,0.053601,0.014699
3,DummyClassifier,4,0.100500,0.5,0.0,0.0,0.0,0.055243,0.014474
4,DummyClassifier,5,0.100500,0.5,0.0,0.0,0.0,0.051567,0.014848


## 6. Speicherung der Baseline-Ergebnisse

In [9]:
dummy_results_path = (
    results_dir / "dummy_classifier_cv.csv"
)

dummy_scores.to_csv(
    dummy_results_path,
    index=False
)

print("Gespeichert:", dummy_results_path)

Gespeichert: /home/lenovo/Dokumente/Uni/Ada_Projekt/results/cv/dummy_classifier_cv.csv


In [10]:
dummy_summary = dummy_scores[
    [
        "PR_AUC",
        "ROC_AUC",
        "Precision",
        "Recall",
        "F1"
    ]
].agg(["mean", "std"])

dummy_summary

,PR_AUC,ROC_AUC,Precision,Recall,F1
mean,0.100488,0.5,0.0,0.0,0.0
std,0.000017,0.0,0.0,0.0,0.0
